In [ ]:
import os as _os
import sys as _sys

_root = _os.path.abspath(_os.getcwd())
while _root != _os.path.dirname(_root) and not _os.path.isfile(_os.path.join(_root, "eval_metrics.py")):
    _root = _os.path.dirname(_root)
if _root not in _sys.path:
    _sys.path.insert(0, _root)
from eval_metrics import evaluate, summary, time_inference, time_inference_torch
from gps_ids import protocol
from gps_ids.results import append_validated

import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import time
import os

In [12]:
# Constants
NUM_CLASSES = 2
RANDOM_STATE = 1082
TRAIN_SIZE = 0.8
DATA_PATH = "../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv"
RESULTS_CSV = "../../../UAVATTACK_Results.csv"
BATCH_SIZE = 64
HIDDEN_SIZE = 64
NUM_EPOCHS = 100
LEARNING_RATE = 0.001
target_col = "label"

In [13]:
# Feature sets
feature_sets = [
    {
        "name": "LSTM Classifier (Explainability Features)",
        "features": [
            'vdop', 'ax', 'hdop', 'cog_rad', 'jamming_indicator',
            'terrain_alt_valid', 'dist_bottom_valid', 'terrain_alt',
            'lat_x', 'lat_y'
        ]
    },
    {
        "name": "LSTM Classifier (XGBoost Features)",
        "features": [
            'lat_y', 'vel_e_m_s', 'vel_n_m_s', 'c_variance_rad', 'hdop',
            's_variance_m_s', 'q_0_', 'vel_m_s', 'cog_rad', 'q_3_'
        ]
    }
]


In [ ]:
def run_lstm_for_feature_set(feature_set_name, feature_cols):
    print(f"Training model: {feature_set_name}")
    exp = protocol.real_heldout("uav", features=feature_cols)
    X_train_scaled = exp.X_train.values
    X_test_scaled = exp.X_test.values
    y_train = exp.y_train
    y_test = exp.y_test

    # Prepare data for LSTM: (samples, sequence_length, input_size)
    INPUT_SIZE = 1
    SEQUENCE_LENGTH = X_train_scaled.shape[1]
    X_train_lstm = X_train_scaled.astype('float32').reshape(-1, SEQUENCE_LENGTH, INPUT_SIZE)
    X_test_lstm = X_test_scaled.astype('float32').reshape(-1, SEQUENCE_LENGTH, INPUT_SIZE)
    y_train_np = y_train.astype('int64')
    y_test_np = y_test.astype('int64')

    # Torch tensors
    X_train_tensor = torch.tensor(X_train_lstm)
    y_train_tensor = torch.tensor(y_train_np)
    X_test_tensor = torch.tensor(X_test_lstm)
    y_test_tensor = torch.tensor(y_test_np)

    # DataLoader
    train_data = TensorDataset(X_train_tensor, y_train_tensor)
    test_data = TensorDataset(X_test_tensor, y_test_tensor)
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

    # Model definition
    class LSTMClassifier(nn.Module):
        def __init__(self, input_size, hidden_size, num_classes):
            super(LSTMClassifier, self).__init__()
            self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True, num_layers=2)
            self.fc = nn.Linear(hidden_size, num_classes)

        def forward(self, x):
            out, _ = self.lstm(x)
            out = out[:, -1, :]
            out = self.fc(out)
            return out

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    np.random.seed(RANDOM_STATE)
    torch.manual_seed(RANDOM_STATE)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(RANDOM_STATE)
        torch.cuda.manual_seed_all(RANDOM_STATE)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    model = LSTMClassifier(INPUT_SIZE, HIDDEN_SIZE, NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # Training
    start_time = time.time()
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * X_batch.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct / total
        if (epoch+1) % 10 == 0 or epoch == 0 or epoch == NUM_EPOCHS-1:
            print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")
    train_time = time.time() - start_time

    # Evaluation
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            all_probs.extend(torch.softmax(outputs, dim=1).cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.numpy())

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, digits=4))
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    print("Accuracy:", accuracy * 100)
    print("F1 Score:", f1 * 100)

    # Save results for this model
    _row = evaluate(
        all_labels,
        all_preds,
        model_name=feature_set_name,
        family="FeatureEngineering",
        y_proba=all_probs,
        train_time_sec=train_time,
        num_epochs=NUM_EPOCHS,
        parameters=list(dict(model.named_parameters()).keys()),
        timing=time_inference_torch(model, next(iter(test_loader))[0].to(device)),
        **exp.eval_kwargs(),
    )
    append_validated(_row, exp.results_csv)
    print(summary(_row))
    print(f"Results appended to {exp.results_csv}\n")

for fs in feature_sets:
    run_lstm_for_feature_set(fs["name"], fs["features"])
